In [ ]:
# First, let's import the libraries to clean this mess.

import pandas as pd
import numpy as np # Just to check the .nan values
import matplotlib.pyplot as plt

In [ ]:
# Now, load the data using pandas, and determine the dataframe.

df = pd.read_csv('ufo_sighting_data.csv', low_memory=False)

df.info() # With the df formed, using this command will show me the type of each column, and most of them are incorrect.

initial_rows = len(df) # this is a variable that will help in the future.

In [ ]:
# Formatting the columns.

# 1. Date_time

df['Date_time'] = pd.to_datetime(df['Date_time'], format='%m/%d/%Y %H:%M', errors='coerce') # With this line, I format it the same way is specified in the .csv file and I include the errors method to avoid any problems and in the case there's a not valid cell, I can replace it or drop it.

# Now, let's verify there's not a large amount of invalid values (NaT or Null) after formatting the column.

df['Date_time'].isna().sum() # With this line, it shows 694 null or NaT values, at this point I can decide to erase it or replace the values, let's calculate the impact of this value.

percentage_cells = df['Date_time'].isna().sum() / df['Date_time'].count() * 100 # By calculating the percentage of cells invalid, I can determine if it's okay to delete them or not.

# print(f'{percentage_cells:.2f}% are invalid') # As we can see, is less than 1% so it's okay to delete them, it won't affect the data greatly.

df = df.dropna(subset=['Date_time']) # This is the line to delete the invalid values, we need to delete the entire na values in the df, but using the column Date_time, that's why I use the subset argument.

df['Date_time'].isna().sum() # And now, the Date_time column has been cleaned effectively.

In [ ]:
# 2. city; this one's simpler than the previous one, because it only needs to be standardized and because there's no nulls, the values '?' and '??' will be replaced for 'unknown'

# Standardization

df['city'] = df['city'].str.lower() # This code is to keep all the letters lowercase.
df['city'] = df['city'].str.strip() # This one is to delete blank spaces at the beginning or end of the strings.

# '?' and '??' values handling.

df['city'] = df['city'].replace('?', 'unknown') # The 'replace' module works is self-explanatory, the first value is the one being replaced, and the second one is the one replacing the first value.
df['city'] = df['city'].replace('??', 'unknown') # Same here.

df['city'].isna().sum()

In [ ]:
# 3. state/province & country; the same as the previous column, some standardization and missing values handling.

# The process was shown in the last column, so this time, a new way will be implemented, it's faster.

states_and_countries = ['state/province', 'country'] # with this list, now is possible to create a for loop.

for stco in states_and_countries: # It will check all the values within the states_and_countries list, compare it with the values within the Dataframe, and replace them automatically using the standardization and filling the missing or na values

    df[stco] = df[stco].str.lower().str.strip() # Convert everything to lower case and strip from blank spaces at the beginning and end of each string.
    df[stco] = df[stco].fillna('unknown') # Replacing missing values or invalid values with 'unknown'

# Now that the loop is finish, let's check if the columns has any missing or invalid values.

df[states_and_countries].isna().sum() # It's clean.

In [ ]:
# 4. UFO_shape; the same as the previous columns.

df['UFO_shape'] = df['UFO_shape'].str.lower().str.strip()
df['UFO_shape'] = df['UFO_shape'].fillna('unknown')

df['UFO_shape'].isna().sum()

In [ ]:
# 5. length_of_encounter_seconds; this column it seems to be the more important one for the numerical analysis, so, beside the things that were already done, this one will be formatted to a numerical type (float), and, it requires more careful treatment.

df['length_of_encounter_seconds'] = pd.to_numeric(df['length_of_encounter_seconds'], errors='coerce') # Again, errors='coerce' will help to force a format in some values that might be invalid or not numerical 'enough' for the transformation.

df.loc[df['length_of_encounter_seconds'].isna()] # By using this line, we can see the only values that are invalid, and by using the 'described_duration_of_encounter' column, we can replace the invalid values more carefully, extracting the index of the three invalid values 27822, 35692, 58591.

df.loc[[27822, 35692, 58591], ['length_of_encounter_seconds', 'described_duration_of_encounter']]

df.loc[[27822], ['length_of_encounter_seconds']] = 2 # 'each a few seconds' doesn't explain anything, but in the original dataset it was 2.
df.loc[[35692], ['length_of_encounter_seconds']] = 8 # It says 8 seconds, so 8 seconds it is.
df.loc[[58591], ['length_of_encounter_seconds']] = 0.5 # It said 1/2 seconds, so 0.5 it is.

df.loc[[27822, 35692, 58591], ['length_of_encounter_seconds', 'described_duration_of_encounter']]

In [ ]:
# 6. described_duration_of_encounter; length_of_encounter_seconds is more accurate and effective than this one, so it'll be deleted.

df = df.drop('described_duration_of_encounter', axis=1) # axis=1 means columns.

df.info() # Check if the column was erased.

In [ ]:
# 7. description; impute the invalid values, and clean some parts of the text.

df['description'] = df['description'].fillna('no description provided')

# In the following lines,some weird texts were replaced, those texts are escape characters from html.

df['description'] = df['description'].str.replace('&#44', ',')
df['description'] = df['description'].str.replace('&amp;', '&')
df['description'] = df['description'].str.replace('&#39', "'")

# Finishing with some standardization.

df['description'] = df['description'].str.lower().str.strip()

df['description'].isna().sum()

In [ ]:
# 8. date_documented; it needs to be formatted as datetime, and of course, handle some missing values if there are any.
df['date_documented'] = pd.to_datetime(df['date_documented'])
df.info() # Checking the new formats.

In [ ]:
# 9 latitude/longitude; let's format them to become float values, and check for any missing values.

num_columns = ['latitude', 'longitude'] # Create a new list, to format them faster using a for loop, again.

for nc in num_columns:

    df[nc] = pd.to_numeric(df[nc], errors='coerce') # Determine the line of code for the loop.

df = df.dropna(subset=['latitude', 'longitude']) # Dropping the missing values, there was only one, so it's not a big deal for the data.

# Range validation; it's necessary to ensure that the coordinate ranges don't exceed their limit.
# for latitude is from -90 to 90, and from longitude is from -180 to 180.

df = df[
    (df['latitude'] >= -90) & (df['latitude'] <= 90) &
    (df['longitude'] >= -180) & (df['longitude'] <= 180)
]

In [ ]:
# 10. Results!

data_types = df.dtypes
sample_of_data = df.head()
remaining_rows = len(df) # This one will be paired with the one originated at the beginning (initial_rows).
rows_lost = initial_rows - remaining_rows

print(f'[Final Data Types] \n{data_types}')
print('') # These two need space.
print(f'[Sample of Cleaned Data] \n{sample_of_data}')
print('')
print(f'[Data Loss Impact]')
print(f'Initial Rows: {initial_rows}')
print(f'Final Rows: {remaining_rows}')
print(f'Percentage of Data Lost: {(rows_lost / initial_rows * 100):.2f}%') # :.2f is to just show to decimal numbers.